Databricks notebook source
MAGIC %md
MAGIC # Bronze: clinical trial ingestion (landing parquet to Delta)

In [0]:
# COMMAND ----------
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog", "clinical_dev")
dbutils.widgets.text("landing_path", "abfss://raw@landingstorageac.dfs.core.windows.net/clinical_trial/")

In [0]:
catalog = dbutils.widgets.get("catalog")
landing = dbutils.widgets.get("landing_path")
ops = f"/Volumes/{catalog}/bronze/ops/clinical_trial_raw"

In [0]:
# COMMAND ----------
df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "parquet")
      .option("cloudFiles.schemaLocation", f"{ops}/schema")
      .load(landing)
      .withColumn("_ingest_time", F.current_timestamp())
      .withColumn("_source_file", F.col("_metadata.file_path")))

In [0]:
(df.writeStream
   .option("checkpointLocation", f"{ops}/checkpoint")
   .option("mergeSchema", "true")
   .trigger(availableNow=True)
   .toTable(f"{catalog}.bronze.clinical_trial_raw"))

In [0]:
target_table = f"{catalog}.bronze.clinical_trial_raw"

In [0]:
count = spark.table(target_table).count()

In [0]:
print(f"Total rows in Bronze table: {count}")

In [0]:
spark.table(target_table).display()

In [0]:
%sql
SELECT COUNT(*) AS rows_with_rescued_data
FROM clinical_dev.bronze.clinical_trial_raw
WHERE _rescued_data IS NOT NULL;